In [1]:
import pandas as pd
import os
import re
import numpy as np
import hdbscan
from sentence_transformers import SentenceTransformer
import umap
import plotly.express as px
from sklearn.decomposition import PCA

import stanza

# Завантажуємо та ініціалізуємо українську модель
stanza.download("uk")
nlp = stanza.Pipeline("uk", processors="tokenize,lemma")

from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 90% !important; }</style>"))

C:\ProgramData\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-25 23:44:59 INFO: Downloaded file to C:\Users\roman.y.melnyk\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\resources.json
2026-07-25 23:44:59 INFO: Downloading default packages for language: uk (Ukrainian) ...
2026-07-25 23:45:01 INFO: File exists: C:\Users\roman.y.melnyk\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\uk\default.zip
2026-07-25 23:45:06 INFO: Finished downloading models and saved to C:\Users\roman.y.melnyk\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources
2026-07-25 23:45:06 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-07-25 23:45:07 INFO: Dow

In [24]:
df = pd.read_csv("data_extractor/wiki_data_ukr_clustering_new.csv").fillna('')
df = df[~df['embedding_text'].str.contains(r'[a-zA-Z]', na=False)]

print('Df size:', df.shape)

Df size: (39803, 6)


In [25]:
df = df[df.embedding_text.str.len()>30]
print('Df size:', df.shape)

Df size: (14805, 6)


In [26]:
pd.set_option('display.max_colwidth', None)

df[['PersonName','Description']].head(10)

,PersonName,Description
0,Ігор Рюрикович,князь Київський 912922 по 944 років
2,Ярослав Мудрий,"український державний діяч часів Київської Русі, великий князь київський, родоначальник багатьох сучасних монархічних династій у Європі"
3,Святополк Володимирович,"Великий князь київський, представник династії Рюриковичів"
5,Судислав Володимирович,"Князь Пскову, молодший син Володимира Святославича"
7,Добронега Володимирівна,"королева Польщі, київська княжна"
8,Анастасія Ярославна,"княжна київська, королева Угорщини"
10,Святослав Ярославич,"руський князь із династії Рюриковичів, князь чернігівський і великий князь київський"
14,Святополк II Ізяславич,"Великий князь київський, князь полоцький, новгородський і турівський"
15,Володимир Мономах,"Великий князь київський, князь чернігівський і переяславський"
17,Нестор-літописець,давньоруський літописець та письменник-агіограф


In [103]:
def safe(v):
    if v is None:
        return None
    if isinstance(v, float) and pd.isna(v):
        return None
    v = str(v).strip()
    return v if v else None


def build_embedding_text(row):
    """
    Покращена версія:
    - Дедуплікація Description vs Occupation (Description додається лише якщо несе іншу інформацію)
    - Додано поле Position (раніше не включалось)
    - Видалено BirthPeriod і BirthPlace — вони були географічним шумом, що заважав
      кластеризації за діяльністю (70%+ мали "радянської доби" і тягнули вектори разом)
    - Видалено шаблонні маркери "Основна діяльність:", "Професія:" тощо —
      вони займали токени без семантичного навантаження для трансформера
    """
    parts = []

    desc = safe(row.get("Description"))
    occ  = safe(row.get("Occupation"))

    # Дедуплікація: додаємо Description лише якщо він НЕ є підрядком Occupation
    if desc and occ:
        if desc.lower() not in occ.lower():
            parts.append(desc)
        parts.append(occ)
    elif occ:
        parts.append(occ)
    elif desc:
        parts.append(desc)

    if safe(row.get("Field")):
        parts.append(row["Field"])

    if safe(row.get("Position")):   # ← НОВЕ: раніше не включалось
        parts.append(row["Position"])

    if safe(row.get("Politics")):
        parts.append(row["Politics"])

    # BirthPeriod і BirthPlace НЕ включаємо в embedding — залишаються як metadata колонки
    return ". ".join(parts)


ISO_Z_REGEX = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")


def ukrainian_cultural_period(date: str | None) -> str | None:
    if not date or not ISO_Z_REGEX.match(str(date)):
        return None
    year = int(str(date)[:4])
    if year < 1300:
        return "давньоруської доби"
    elif year < 1600:
        return "ранньомодерної доби"
    elif year < 1800:
        return "козацької доби"
    elif year < 1917:
        return "імперської індустріальної доби"
    elif year < 1922:
        return "революційної доби"
    elif year < 1991:
        return "радянської доби"
    else:
        return "сучасної України"


def clean_text(t: str) -> str:
    return (
        t.replace(" ,", ",")
         .replace(" .", ".")
         .replace("  ", " ")
         .strip()
    )

def deduplicate(row):
    # Замінюємо коми на пробіли, перетворюємо на нижній регістр (для точного порівняння)
    # та розбиваємо на окремі слова
    words = row.replace(",", " ").split()

    # Створюємо список унікальних слів, зберігаючи початковий порядок їхньої появи
    seen = set()
    unique_words = []
    for word in words:
        # Можна додатково очистити слово від крапок чи інших знаків, якщо потрібно
        clean_word = word.strip().lower()
        if clean_word not in seen:
            seen.add(clean_word)
            unique_words.append(
                word
            )  # додаємо оригінальне слово (зберігаючи регістр, якщо треба)

    # Об'єднуємо слова назад у речення (наприклад, через кому чи пробіл)
    return ",".join(unique_words)

In [104]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer(lang="uk")

STOP_WORDS = {"та", "і", "й", "з", "в", "на", "у", "для", "по"}

# 2. Ваші специфічні слова для видалення (національності/маркери)
# Список прикметників для видалення
GEOGRAPHIC_WORDS = {
    "український",
    "українська",
    "радянський",
    "радянська",
    "російський",
    "російська",
    "білоруський",
    "білоруська",
    "кримськотатарський",
    "кримськотатарська",
    "польський",
    "польська",
}


def get_lemma_fixed(word):
    word_lower = word.strip().lower()
    parsed = morph.parse(word_lower)
    if not parsed:
        return word_lower

    p = parsed[0]
    lemma = p.normal_form

    if p.tag.POS == "NOUN" and p.tag.gender == "femn" and lemma.endswith("ка"):
        base = lemma[:-2]
        if base.endswith("ч"):
            potential_masc = base[:-1] + "к"
        elif base.endswith("т"):
            potential_masc = base
        else:
            potential_masc = base

        if morph.word_is_known(potential_masc):
            return potential_masc
    return lemma


def clean_and_restore_text(text):
    if not text or not isinstance(text, str):
        return ""

    # 1. Тимчасово прибираємо коми, відновлюючи нормальні пробіли між словами
    # Замість "польський,хімік,інженер" отримуємо "польський хімік інженер"
    clean_text = text.replace(",", " ")
    words = clean_text.split()

    seen_lemmas = set()
    final_words = []

    for i, word in enumerate(words):
        word_clean = word.strip()
        word_lower = word_clean.lower()

        if not word_clean:
            continue

        # 2. Видаляємо гео-прикметники (польський, українська тощо)
        if word_lower in GEOGRAPHIC_WORDS:
            continue

        # 3. Видаляємо службові слова ("та", "і"), якщо вони зависли на початку
        # або йдуть після видаленого слова
        if word_lower in {"та", "і", "й", "до"} and len(final_words) == 0:
            continue

        # 4. Дедуплікація фемінітивів (співачка/співак)
        lemma = get_lemma_fixed(word_clean)
        if lemma in seen_lemmas:
            # Якщо це точний дублікат або фемінітив, який вже був — пропускаємо
            continue
        else:
            seen_lemmas.add(lemma)

        # Зберігаємо оригінальний регістр слова (наприклад, для "Сейму", "Верховного")
        final_words.append(word_clean)

    if not final_words:
        return ""

    # 5. Зшиваємо в ОДНЕ зв'язне речення через пробіл (без ком після кожного слова!)
    result = " ".join(final_words)

    # За бажанням, якщо у вас всередині тексту є стійкі переліки через кому,
    # їх можна повернути, але для ваших прикладів чистий текст через пробіл підходить найкраще.

    # Робимо першу літру великою
    return result[0].upper() + result[1:]

In [105]:
# --- Застосувати ---
#df['BirthPeriod'] = df.BirthDate.apply(ukrainian_cultural_period)
df["embedding_text"] = df.apply(build_embedding_text, axis=1)
df["embedding_text"] = [clean_text(x) for x in df.embedding_text]
df.embedding_text = [re.sub(r'[.;|]', ',', text) for text in df.embedding_text]

df["embedding_text"] = [deduplicate(x) for x in df.embedding_text]

df["embedding_text"] = [clean_and_restore_text(x) for x in df.embedding_text]

print(df.shape)

(14805, 6)


In [106]:
pd.set_option('display.max_colwidth', None)

df[['PersonName','Description','embedding_text']].head()

,PersonName,Description,embedding_text
0,Ігор Рюрикович,князь Київський 912922 по 944 років,Князь Київський 912922 по 944 років
2,Ярослав Мудрий,"український державний діяч часів Київської Русі, великий князь київський, родоначальник багатьох сучасних монархічних династій у Європі",Державний діяч часів Київської Русі великий князь родоначальник багатьох сучасних монархічних династій у Європі
3,Святополк Володимирович,"Великий князь київський, представник династії Рюриковичів",Великий князь київський представник династії Рюриковичів
5,Судислав Володимирович,"Князь Пскову, молодший син Володимира Святославича",Князь Пскову молодший син Володимира Святославича
7,Добронега Володимирівна,"королева Польщі, київська княжна",Королева Польщі київська княжна


In [107]:
#words_to_remove = [
#    "український", "українська",
#    "радянський", "радянська",
#    "російський", "російська",
#    "білоруський", "білоруська",
#    "кримськотатарський", "кримськотатарська"
#]
#pattern = r"\b(" + "|".join(words_to_remove) + r")\b"
#df["embedding_text"] = df["embedding_text"].str.replace(pattern, "", regex=True)
#df["embedding_text"] = df["embedding_text"].str.strip()

In [108]:
### Most frequent words

from itertools import chain

dd = list(chain.from_iterable(x.replace(", ", " ").split() for x in df.embedding_text))

In [109]:
pd.Series(dd).value_counts().head(10)

діяч                   2212
і                      2123
та                     1708
Військовослужбовець    1467
України                 994
громадський             942
педагог                 784
у                       490
походження              459
Письменник              413
Name: count, dtype: int64

In [110]:
df.embedding_text.head().tolist()

['Князь Київський 912922 по 944 років',
 'Державний діяч часів Київської Русі великий князь родоначальник багатьох сучасних монархічних династій у Європі',
 'Великий князь київський представник династії Рюриковичів',
 'Князь Пскову молодший син Володимира Святославича',
 'Королева Польщі київська княжна']

### Create Embeddings

In [111]:
from sentence_transformers import SentenceTransformer

#model = SentenceTransformer("intfloat/multilingual-e5-large")
model = SentenceTransformer("intfloat/multilingual-e5-small")

In [112]:
texts = ["passage: " + t for t in df["embedding_text"]]

pool = model.start_multi_process_pool()

print(f"Started processing {len(texts)} rows...")

embeddings = model.encode(
    texts,
    batch_size=64,    
    chunk_size=None,
    show_progress_bar=True,
    normalize_embeddings=True
)

model.stop_multi_process_pool(pool)

print("Embeddings done!")

Started processing 14805 rows...


Batches: 100%|█████| 232/232 [00:46<00:00,  5.00it/s]

Embeddings done!


In [113]:
### Cache data
df["embeddings"] = embeddings.tolist()
df.to_parquet("wikidata_with_embeddings_lema.parquet")

### Reduce dimensionality for clustering

In [15]:
### Read cached data
df = pd.read_parquet("wikidata_with_embeddings_lema.parquet")
embeddings = np.array(df["embeddings"].tolist())
print('Embedding dimension:', len(embeddings[0]))
print('Data set size:', len(embeddings))
df = df.drop('embeddings', axis=1)

Embedding dimension: 384
Data set size: 14805


In [11]:
# 1. UMAP: Створюємо простір для кластеризації
# Для e5-small з її 384 вимірами n_neighbors=20-30 є оптимальним

# Звичайний UMAP на фінальній карті може намалювати їх як два однакові за розміром кружечки. Він розтягне Монако і стисне Рівне, щоб вони виглядали «гарно». 
# densmap=True змушує UMAP пам'ятати, що Монако було дуже щільним, а Рівне — розрідженим. На карті Монако залишиться маленькою цяткою, а Рівне — великою хмарою.

reducer_labels = umap.UMAP(
    n_neighbors=70,        # Збалансовано: не надто дрібні, не надто великі кластери. Більше сусідів = більші кластери
    n_components=8,        # Стискаємо до 10 вимірів для HDBSCAN
    metric='cosine',        # E5 моделі найкраще працюють з косинусною відстанню
    min_dist=0.0,           # Максимальна щільність для кластеризації. Для кластеризації нам не потрібна "красива" картинка, нам потрібно, щоб точки з однаковим змістом лежали буквально одна на одній.
    #densmap=True,
    random_state=42,
    low_memory=True,        # Економить RAM, актуально для великих датасетів
    n_jobs=-1,              # Використовує всі доступні ядра процесора
    #init='pca',            # Швидша ініціалізація, ніж випадкова (spectral)
    verbose=True
)
embeddings_for_clustering = reducer_labels.fit_transform(embeddings)

C:\ProgramData\Python313\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=8, n_jobs=1, n_neighbors=70, random_state=42, verbose=True)
Mon Jul 20 21:59:16 2026 Construct fuzzy simplicial set
Mon Jul 20 21:59:16 2026 Finding Nearest Neighbors
Mon Jul 20 21:59:16 2026 Building RP forest with 11 trees
Mon Jul 20 21:59:17 2026 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	Stopping threshold met -- exiting after 4 iterations
Mon Jul 20 21:59:45 2026 Finished Nearest Neighbor Search
Mon Jul 20 21:59:46 2026 Construct embedding


Epochs completed:   1%| ▋                                                               2/200 [00:00]

	completed  0  /  200 epochs


Epochs completed:  10%| ██████▌                                                        21/200 [00:06]

...

	completed  160  /  200 epochs


Epochs completed:  90%| ███████████████████████████████████████████████████████▏      181/200 [01:01]

	completed  180  /  200 epochs


Epochs completed: 100%| █████████████████████████████████████████████████████████████ 200/200 [01:11]

Mon Jul 20 22:01:03 2026 Finished embedding


### Clustering with HDBSCAN

In [12]:
# 2. HDBSCAN: Шукаємо кластери
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=70,                # Мінімальна кількість людей у професійній групі. Збільш до 200–500 (або навіть до 1000, якщо хочеш бачити лише "гігантів"). Це автоматично поглине всі дрібні групи, які раніше були окремими кластерами.
    min_samples=15,                      # Чим більше, тим жорсткіша кластеризація (менше шуму). "Якщо навколо тебе немає ще n друзів, ти — шум, а не частина кластера". Це "обрізає" тонкі хвости у черв'яків.
    cluster_selection_epsilon=0.1,       # Допомагає об'єднувати близькі мікро-групи. Це "дистанція злиття". Якщо два великі кластери знаходяться на цій відстані один від одного, вони стануть одним цілим.
    metric='euclidean',                  # Після UMAP завжди використовуємо euclidean
    cluster_selection_method='eom',
    core_dist_n_jobs=-1
)
df['cluster'] = clusterer.fit_predict(embeddings_for_clustering)
df["cluster_prob"] = clusterer.probabilities_

In [13]:
PROB_THRESHOLD = 0.3

df_core = df[(df["cluster"] != -1) & (df["cluster_prob"] >= PROB_THRESHOLD)].copy()

df_noise = df[(df["cluster"] == -1) | (df["cluster_prob"] < PROB_THRESHOLD)].copy()

print('Clusters number:', df.cluster.nunique())
print("Core points:", len(df_core))
print("Noise / weak points:", len(df_noise))
print("Covered: ", 100*len(df_core)/len(df))
#62

Clusters number: 47
Core points: 9995
Noise / weak points: 4810
Covered:  67.51097602161433


### Reduce dimensionality to 2D for visualization

In [14]:
# 3. UMAP 2D: Для візуалізації. spread=1.5, min_dist=0.01
reducer_2d = umap.UMAP(n_neighbors=100,        # Має бути схожим на параметр для кластеризації або більшим
                       n_components=2, 
                       metric='euclidean', 
                       min_dist=1,           # Додаємо трохи "повітря", щоб кластери не були точками 
                       spread=3,            # Зменшуємо загальний розкид
                       #densmap=True,          # Зберігає локальну щільність
                       target_weight=0.75,      # КРИТИЧНО: чим ближче до 1.0, тим сильніше злипаються однакові кластери
                       random_state=42,
                       low_memory=True,      
                       n_jobs=-1,              
                       init='pca',         
                       verbose=True)

#embeddings_2d = reducer_2d.fit_transform(embeddings_for_clustering, y=df['cluster'])
embeddings_2d = reducer_2d.fit_transform(embeddings, y=df['cluster'])
df['x'], df['y'] = embeddings_2d[:, 0], embeddings_2d[:, 1]

UMAP(init='pca', min_dist=1, n_jobs=1, n_neighbors=100, random_state=42, spread=3, target_weight=0.75, verbose=True)
Mon Jul 20 22:01:05 2026 Construct fuzzy simplicial set
Mon Jul 20 22:01:05 2026 Finding Nearest Neighbors
Mon Jul 20 22:01:05 2026 Building RP forest with 11 trees


C:\ProgramData\Python313\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Mon Jul 20 22:01:06 2026 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	Stopping threshold met -- exiting after 4 iterations
Mon Jul 20 22:02:07 2026 Finished Nearest Neighbor Search
Mon Jul 20 22:02:08 2026 Construct embedding


Epochs completed:   2%| █▎                                                              4/200 [00:01]

	completed  0  /  200 epochs


...

	completed  160  /  200 epochs


Epochs completed:  90%| ███████████████████████████████████████████████████████▏      181/200 [01:22]

	completed  180  /  200 epochs


Epochs completed: 100%| █████████████████████████████████████████████████████████████ 200/200 [01:30]

Mon Jul 20 22:03:41 2026 Finished embedding


In [ ]:
def generate_cluster_names(df):
    topic_names = {}
    # Ігноруємо шум (кластер -1)
    unique_clusters = [c for c in df['cluster'].unique() if c != -1]
    
    for cluster_id in unique_clusters:
        subset = df[df['cluster'] == cluster_id]
        
        # Беремо топ-3 найчастіші професії (Occupation)
        # Розбиваємо рядки, якщо там декілька професій через кому
        all_occs = subset['Occupation'].str.split(', ').explode()
        top_occs = all_occs.value_counts().head(3).index.tolist()
        
        # Формуємо назву: "ID | Професія1 | Професія2"
        name = f"{cluster_id} | " + " | ".join(top_occs)
        topic_names[cluster_id] = name
        
    topic_names[-1] = "-1 | Шум / Невизначені"
    return topic_names

#cluster_names = generate_cluster_names(df_core)
#df_core['topic_name'] = df_core['cluster'].map(cluster_names)

def snap_outliers_to_clusters(df, x_col='x', y_col='y', cluster_col='cluster', threshold_p=80):
    """
    threshold_p: перцентиль відстані. Точки, що далі за цей відсоток від центру, будуть переміщені.
    """
    df_refined = df.copy()
    # Переконаємося, що типи збігаються (наприклад, перетворимо в int, якщо там рядки)
    unique_clusters = [c for c in df[cluster_col].unique() if c != -1]
    
    total_moved = 0
    
    for cluster_id in unique_clusters:
        mask = df_refined[cluster_col] == cluster_id
        if not mask.any(): continue
        
        cluster_points = df_refined.loc[mask, [x_col, y_col]].values
        center = np.median(cluster_points, axis=0)
        distances = np.linalg.norm(cluster_points - center, axis=1)
        
        limit = np.percentile(distances, threshold_p)
        outliers_in_cluster_mask = distances > limit
        
        if np.any(outliers_in_cluster_mask):
            n_outliers = np.sum(outliers_in_cluster_mask)
            # Отримуємо реальні індекси рядків у вихідному DataFrame
            target_indices = df_refined[mask].index[outliers_in_cluster_mask]
            
            jitter = np.random.normal(0, limit * 0.1, size=(n_outliers, 2))
            new_coords = (center + jitter).astype(df_refined[x_col].dtype)
            
            df_refined.loc[target_indices, [x_col, y_col]] = new_coords
            total_moved += n_outliers

    print(f"Готово! Переміщено точок: {total_moved}")
    return df_refined

def snap_outliers_to_clusters_std(df, x_col='x', y_col='y', cluster_col='cluster', n_std=2.0):
    """
    df: вихідний DataFrame
    n_std: поріг у кількості стандартних відхилень. 
           2.0 — перемістить ~5% крайніх точок (якщо розподіл нормальний).
           3.0 — перемістить лише дуже далекі викиди (~0.3%).
    """
    df_refined = df.copy()
    unique_clusters = [c for c in df[cluster_col].unique() if str(c) != '-1' and str(c) != 'noise']
    
    total_moved = 0
    
    for cluster_id in unique_clusters:
        # Створюємо маску для конкретного кластера
        mask = df_refined[cluster_col] == cluster_id
        if not mask.any(): continue
        
        # Отримуємо координати точок кластера
        points = df_refined.loc[mask, [x_col, y_col]]
        
        # 1. Обчислюємо Mean (середнє) та STD (стандартне відхилення)
        mu_x, mu_y = points[x_col].mean(), points[y_col].mean()
        std_x, std_y = points[x_col].std(), points[y_col].std()
        
        # Захист від кластерів з однієї точки або нульовим STD
        if std_x == 0 or std_y == 0 or np.isnan(std_x): continue

        # 2. Визначаємо викиди за Z-score для обох осей
        # Точка вважається викидом, якщо вона далека від центру хоча б за однією віссю
        outliers_mask = (
            (np.abs(points[x_col] - mu_x) > n_std * std_x) | 
            (np.abs(points[y_col] - mu_y) > n_std * std_y)
        )
        
        if outliers_mask.any():
            n_outliers = outliers_mask.sum()
            target_indices = points.index[outliers_mask]
            
            # 3. Генеруємо нові координати навколо середнього значення
            # Додаємо невеликий jitter, пропорційний до STD (наприклад, 10% від відхилення)
            jitter_x = np.random.normal(0, std_x * 0.1, size=n_outliers)
            jitter_y = np.random.normal(0, std_y * 0.1, size=n_outliers)
            
            new_x = (mu_x + jitter_x).astype(df_refined[x_col].dtype)
            new_y = (mu_y + jitter_y).astype(df_refined[y_col].dtype)
            
            # Записуємо нові координати
            df_refined.loc[target_indices, x_col] = new_x
            df_refined.loc[target_indices, y_col] = new_y
            
            total_moved += n_outliers

    print(f"Готово! Переміщено точок за логікою Mean/STD (n={n_std}): {total_moved}")
    return df_refined

def snap_outliers_to_clusters_iqr(df, x_col='x', y_col='y', cluster_col='cluster', k=1.5):
    """
    df: вихідний DataFrame
    k: коефіцієнт розкиду. 
       1.5 — класичний поріг Тукі для викидів.
       1.0 — зробить кластери ще щільнішими (стягне більше точок).
    """
    df_refined = df.copy()
    unique_clusters = [c for c in df[cluster_col].unique() if str(c) not in ['-1', 'noise']]
    
    total_moved = 0
    
    for cluster_id in unique_clusters:
        mask = df_refined[cluster_col] == cluster_id
        if not mask.any(): continue
        
        # Працюємо з координатами поточного кластера
        points = df_refined.loc[mask, [x_col, y_col]]
        
        # 1. Рахуємо квартилі та IQR окремо для X та Y
        q1 = points.quantile(0.25)
        q3 = points.quantile(0.75)
        iqr = q3 - q1
        
        # Визначаємо межі "нормальності"
        lower_bound = q1 - k * iqr
        upper_bound = q3 + k * iqr
        
        # 2. Визначаємо точки, що вийшли за межі хоча б по одній осі
        outliers_mask = (
            (points[x_col] < lower_bound[x_col]) | (points[x_col] > upper_bound[x_col]) |
            (points[y_col] < lower_bound[y_col]) | (points[y_col] > upper_bound[y_col])
        )
        
        if outliers_mask.any():
            n_outliers = outliers_mask.sum()
            target_indices = points.index[outliers_mask]
            
            # 3. Визначаємо центр для приземлення (медіана - найбезпечніша)
            median_center = points.median()
            
            # Створюємо jitter на основі IQR (щоб масштаб шуму відповідав масштабу кластера)
            # Використовуємо 5% від IQR для розкиду навколо центру
            jitter_x = np.random.normal(0, iqr[x_col] * 0.05, size=n_outliers)
            jitter_y = np.random.normal(0, iqr[y_col] * 0.05, size=n_outliers)
            
            new_x = (median_center[x_col] + jitter_x).astype(df_refined[x_col].dtype)
            new_y = (median_center[y_col] + jitter_y).astype(df_refined[y_col].dtype)
            
            # Оновлюємо координати
            df_refined.loc[target_indices, x_col] = new_x
            df_refined.loc[target_indices, y_col] = new_y
            
            total_moved += n_outliers

    print(f"Готово! Переміщено точок за логікою IQR (k={k}): {total_moved}")
    return df_refined
    
df_core = df.loc[df_core.index]
df_noise = df.loc[df_noise.index]

df_core["cluster_label"] = df_core["cluster"].astype(str)
df_noise["cluster_label"] = "noise"

df_core.loc[df_core['Description'].isna(), 'Description'] = ''
#df_core.loc[df_core['Occupation'].isna(), 'Occupation'] = ''


#df_core2 = snap_outliers_to_clusters(df_core, threshold_p=95) 
#df_core2 = snap_outliers_to_clusters_std(df=df_core, x_col='x', y_col='y', cluster_col='cluster', n_std=2.0) 
df_core2 = snap_outliers_to_clusters_iqr(df=df_core, x_col='x', y_col='y', cluster_col='cluster', k=2)

df_core2.to_csv('data/clutering_data.csv', index=False)

In [2]:
df_core = pd.read_csv('data/clutering_data.csv')
df_core2 = df_core.copy()

In [3]:
df_core.head(2)

,PersonName,BirthPlace,Description,WikipediaURL,WikiText,embedding_text,cluster,cluster_prob,x,y,cluster_label
0,Ярослав Мудрий,Київ,український державний діяч часів Київської Рус...,https://uk.wikipedia.org/wiki/Ярослав_Мудрий,ярослав мудрий український державний діяч часі...,Державний діяч часів Київської Русі великий кн...,37,0.865784,14.803871,11.974056,37
1,Євпраксія Всеволодівна,Київ,імператорка Священної Римської імперії,https://uk.wikipedia.org/wiki/Євпраксія_Всевол...,євпраксія всеволодівна імператорка священної р...,Імператорка Священної Римської імперії,34,0.774990,-6.877429,11.725318,34


In [5]:
import plotly.express as px
import plotly.io as pio

sorted_labels = sorted(df_core["cluster_label"].unique())

x_min = df_core2["x"].min()
x_max = df_core2["x"].max()
y_min = df_core2["y"].min()
y_max = df_core2["y"].max()

x_center = (x_min + x_max) / 2
y_center = (y_min + y_max) / 2

x_span = x_max - x_min
y_span = y_max - y_min

zoom_factor = 1.5


# 1. Створення графіка
fig = px.scatter(
    df_core2,
    x="x",
    y="y",
    color="cluster_label",
    category_orders={"cluster_label": sorted_labels},
    hover_name="PersonName",
    custom_data=["WikipediaURL", "cluster_label", "Description"],
    template="plotly_dark",
    render_mode="webgl",
    color_discrete_sequence=px.colors.qualitative.Alphabet
)

fig.update_traces(
    marker=dict(
        size=4,
        opacity=0.75,
        line=dict(width=0)
    ),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Кластер: %{customdata[1]}<br>"
        "Опис: %{customdata[2]}<br>"
        "<b>Клікніть, щоб відкрити Wiki</b>"
        "<extra></extra>"
    )
)

# 2. Layout без фіксованої висоти
fig.update_layout(
    autosize=True,

    title=dict(
        #text="🇺🇦 Кластеризація українців у Вікіпедії",
        font=dict(size=18, color="white"),
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top"
    ),

    paper_bgcolor="#111111",
    plot_bgcolor="#111111",

    font=dict(
        color="white",
        family="Segoe UI, Arial, sans-serif"
    ),

    # Не задавайте height=800
    margin=dict(l=0, r=0, t=50, b=0, pad=0),
    coloraxis_colorbar=dict(title=""),
    legend=dict(
        title_text="",
        bgcolor="rgba(30,30,30,0.85)",
        bordercolor="#444",
        borderwidth=1,
        font=dict(size=10),
        itemsizing="constant",
        tracegroupgap=2
    ),

    dragmode="pan",
    hovermode="closest",

    xaxis=dict(
        range=[
            x_center - x_span * zoom_factor / 2,
            x_center + x_span * zoom_factor / 2
        ],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        automargin=False,
        title=''
    ),

    yaxis=dict(
        range=[
            y_center - y_span * zoom_factor / 2,
            y_center + y_span * zoom_factor / 2
        ],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        automargin=False,
        title=''
    )
)

# 3. JavaScript для кліку по точці
post_script = """
const plot = document.getElementById('{plot_id}');
const lang = new URLSearchParams(window.location.search).get('lang') || localStorage.getItem('wikimap_lang') || 'ua';
const translations = {
    ua: {
        pageTitle: 'Кластеризація українців у Вікіпедії',
        chartTitle: '<b>Кластеризація українців у Вікіпедії</b>',
        cluster: 'Кластер', description: 'Опис', click: 'Клікніть, щоб відкрити Вікіпедію', legend: 'Кластер'
    },
    en: {
        pageTitle: 'Ukrainians on Wikipedia clustering',
        chartTitle: '<b>Ukrainians on Wikipedia clustering</b>',
        cluster: 'Cluster', description: 'Description', click: 'Click to open Wikipedia', legend: 'Cluster'
    }
};
const T = translations[lang] || translations.ua;
document.documentElement.lang = lang === 'en' ? 'en' : 'uk';
document.title = T.pageTitle;

Plotly.relayout(plot, {
    'title.text': T.chartTitle,
    'legend.title.text': T.legend
});
Plotly.restyle(plot, {
    hovertemplate: '<b>%{hovertext}</b><br>' +
        T.cluster + ': %{customdata[1]}<br>' +
        T.description + ': %{customdata[2]}<br>' +
        '<b>' + T.click + '</b><extra></extra>'
});


plot.on('plotly_click', function(data) {
    if (!data.points || !data.points.length) {
        return;
    }

    const customData = data.points[0].customdata;
    const url = customData ? customData[0] : null;

    if (url && /^https?:\\/\\//i.test(url)) {
        window.open(url, '_blank', 'noopener,noreferrer');
    }
});

/*
 * Примусово перераховуємо розмір після завантаження сторінки.
 */
requestAnimationFrame(function() {
    Plotly.Plots.resize(plot);
});
"""

# Plotly div
plot_html = pio.to_html(
    fig,
    full_html=False,
    include_plotlyjs="cdn",
    config={
        "scrollZoom": True,
        "responsive": True,
        "displaylogo": False
    },
    default_width="100%",
    default_height="100%",
    div_id="cluster_plot",
    post_script=post_script
)

# 4. Повноекранна HTML-сторінка
html_document = f"""<!DOCTYPE html>
<html lang="uk">
<head>
    <meta charset="UTF-8">

    <!-- Google tag (gtag.js) -->
    <script async src='https://www.googletagmanager.com/gtag/js?id=G-NRQ1JX8D3W'></script>
    <script>
        window.dataLayer = window.dataLayer || [];
        function gtag(){{dataLayer.push(arguments);}}
        gtag('js', new Date());
        gtag('config', 'G-NRQ1JX8D3W');
    </script>

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <style>
        * {{
            box-sizing: border-box;
        }}

        html,
        body {{
            width: 100%;
            height: 100%;
            margin: 0;
            padding: 0;
            border: 0;
            overflow: hidden;
            background: #111111;
        }}

        #cluster_plot {{
            position: fixed !important;
            inset: 0 !important;

            width: 100vw !important;
            height: 100vh !important;
            height: 100dvh !important;

            margin: 0 !important;
            padding: 0 !important;
            border: 0 !important;
            /* Plotly needs direct pointer events for pinch-to-zoom on phones. */
            touch-action: none !important;
        }}

        #cluster_plot .plot-container,
        #cluster_plot .svg-container {{
            width: 100% !important;
            height: 100% !important;
            touch-action: none !important;
        }}
    </style>
</head>

<body>
    {plot_html}

    <script>
        /*
         * Оновлюємо графік при зміні розміру вікна,
         * переході в повноекранний режим або зміні орієнтації.
         */
        const resizePlot = () => {{
            const plot = document.getElementById("cluster_plot");

            if (plot && window.Plotly) {{
                Plotly.Plots.resize(plot);
            }}
        }};

        window.addEventListener("load", resizePlot);
        window.addEventListener("resize", resizePlot);
        window.addEventListener("orientationchange", resizePlot);
        if (window.visualViewport) {{
            window.visualViewport.addEventListener("resize", resizePlot);
        }}
        document.addEventListener("fullscreenchange", resizePlot);
    </script>
</body>
</html>
"""

# 5. Збереження
with open("data/clustering_map.html", "w", encoding="utf-8") as file:
    file.write(html_document)